### CONEXIÓN CON SQL SERVER 

In [1]:
!pip install sqlalchemy
!pip install pyodbc

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   -------------- ------------------------- 0.8/2.1 MB 2.0 MB/s eta 0:00:01
   ------------------- -------------------- 1.0/2.1 MB 2.0 MB/s eta 0:00:01
   ------------------------ --------------- 1.3/2.1 MB 1.9 MB/s eta 0:00:01
   ----------------------------- ---------- 1.6/2.1 MB 1.7 MB/s eta 0:00:01
   ---------------------------------------  2.1/2.1 MB 1.8 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 1.7 MB/s  0:00:01

   ---------------------------------------- 0/2 [greenlet]
   ---------------------------------------- 0/2 [greenlet]
   ---------------------------------------- 0/2 [greenlet]
   ---------------------------------------- 0/2 [greenlet]
   ---------------------------------------- 0/2 [greenlet]
   ---------------------------------------- 0/2 [greenlet]
   ---------------------------------------- 0/2 [green


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
import pandas as pd
import pyodbc

from sqlalchemy import create_engine

In [37]:
import pandas as pd

gold = pd.read_csv(
    r"C:\Users\USUARIO\Documents\GitHub\Empresas-Colombianas\data\Gold\empresas_colombia_gold_Final.csv"
)

gold.head()

,ranking_2025,nit,razon_social,región,departamento_domicilio,ciudad_domicilio,macrosector,ingresos_operacionales_2025,ganancia_pérdida_2025,total_activos_2025,total_pasivos_2025,total_patrimonio_2025,crecimiento_ingresos_pct,margen_utilidad_pct,endeudamiento_pct,roa_pct,roe_pct
0,1,899999068,ECOPETROL S.A,BOGOTÁ-CUNDINAMARCA,DISTRITO CAPITAL DE BOGOTÁ,BOGOTA D.C.,MINERO,100590667613,9028764859,198110860034,114350599639,83760260395,-11.703174,8.975748,57.720510,4.557431,10.779294
1,2,830095213,ORGANIZACIÓN TERPEL S.A.,BOGOTÁ-CUNDINAMARCA,DISTRITO CAPITAL DE BOGOTÁ,BOGOTA D.C.,COMERCIO,26395418332,628485892,8112318463,4699968205,3412350258,6.639487,2.381042,57.936190,7.747303,18.417977
2,3,900112515,REFINERIA DE CARTAGENA S.A.,CARIBE,BOLÍVAR,CARTAGENA,MANUFACTURA,22395204745,-821398601,34946245504,11566291299,23379954205,-13.401343,-3.667743,33.097379,-2.350463,-3.513260
3,4,900276962,D1 S A S,BOGOTÁ-CUNDINAMARCA,DISTRITO CAPITAL DE BOGOTÁ,BOGOTA D.C.,COMERCIO,21606607933,418933423,6939606162,6724638908,214967254,11.148555,1.938913,96.902313,6.036847,194.882437
4,5,890904996,EMPRESAS PÚBLICAS DE MEDELLÍN E.S.P.,ANTIOQUIA,ANTIOQUIA,MEDELLIN,SERVICIOS,20285629973,4875861363,71083399625,36111926922,34971472703,-1.403910,24.036036,50.802194,6.859353,13.942396


In [38]:
import pyodbc

conexion = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=LAPTOP-KJGERIIR;"
    "DATABASE=EmpresasColombianas;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

print("Conexión exitosa")

Conexión exitosa


In [39]:
from sqlalchemy import create_engine
import urllib

params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=LAPTOP-KJGERIIR;"
    "DATABASE=EmpresasColombianas;"
    "Trusted_Connection=yes;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

print("Engine creado correctamente")

Engine creado correctamente


In [40]:
pd.read_sql(
    "SELECT name FROM sys.tables",
    engine
)

,name
0,dim_empresa
1,dim_region
2,dim_departamento
3,dim_macrosector
4,fact_indicadores_financieros


### CREACIÓN DE DATAFRAMES DE LAS DIMENCIONES

In [41]:
dim_empresa = (
    gold[["nit", "razon_social"]]
    .drop_duplicates(subset="nit")
    .sort_values("nit")
    .reset_index(drop=True)
)

dim_empresa.head()

,nit,razon_social
0,800000276,AVICOLA EL MADROÑO S.A.
1,800000457,ACOMEDIOS PUBLICIDAD Y MERCADEO LTDA
2,800000750,PETROSANTANDER COLOMBIA GMBH
3,800000946,PROCTER & GAMBLE COLOMBIA LTDA
4,800001845,SERPET JR Y CIA S.A.S.


In [43]:
dim_region = (
    gold[["región"]]
    .drop_duplicates()
    .rename(columns={"región": "region"})
    .sort_values("region")
    .reset_index(drop=True)
)

dim_region.head()   

,region
0,AMAZONÍA
1,ANTIOQUIA
2,BOGOTÁ-CUNDINAMARCA
3,CARIBE
4,CENTRO


In [44]:
dim_departamento = (
    gold[["departamento_domicilio"]]
    .drop_duplicates()
    .rename(columns={"departamento_domicilio": "departamento"})
    .sort_values("departamento")
    .reset_index(drop=True)
)

dim_departamento.head()

,departamento
0,AMAZONAS
1,ANTIOQUIA
2,ARAUCA
3,ATLÁNTICO
4,BOLÍVAR


In [45]:
dim_macrosector = (
    gold[["macrosector"]]
    .drop_duplicates()
    .sort_values("macrosector")
    .reset_index(drop=True)
)

dim_macrosector.head()

,macrosector
0,AGROPECUARIO
1,COMERCIO
2,CONSTRUCCIÓN
3,MANUFACTURA
4,MINERO


Auditoria de las dimenciones

In [46]:
print("Empresas:", dim_empresa.shape)
print("Regiones:", dim_region.shape)
print("Departamentos:", dim_departamento.shape)
print("Macrosectores:", dim_macrosector.shape)

Empresas: (10000, 2)
Regiones: (9, 1)
Departamentos: (32, 1)
Macrosectores: (6, 1)


In [47]:
print(dim_empresa["nit"].duplicated().sum())
print(dim_region["region"].duplicated().sum())
print(dim_departamento["departamento"].duplicated().sum())
print(dim_macrosector["macrosector"].duplicated().sum())

0
0
0
0


Carga de las dimenciones

In [49]:
dim_empresa.to_sql(
    "dim_empresa",
    con=engine,
    if_exists="append",
    index=False
)

print("Dimensión Empresa cargada.")

Dimensión Empresa cargada.


In [50]:
dim_region.to_sql(
    "dim_region",
    con=engine,
    if_exists="append",
    index=False
)

print("Dimensión Región cargada.")

Dimensión Región cargada.


In [51]:
dim_departamento.to_sql(
    "dim_departamento",
    con=engine,
    if_exists="append",
    index=False
)

print("Dimensión Departamento cargada.")

Dimensión Departamento cargada.


In [52]:
dim_macrosector.to_sql(
    "dim_macrosector",
    con=engine,
    if_exists="append",
    index=False
)

print("Dimensión Macrosector cargada.")

Dimensión Macrosector cargada.


In [53]:
dim_empresa_sql = pd.read_sql("""
SELECT id_empresa, nit
FROM dim_empresa
""", engine)

dim_region_sql = pd.read_sql("""
SELECT id_region, region
FROM dim_region
""", engine)

dim_departamento_sql = pd.read_sql("""
SELECT id_departamento, departamento
FROM dim_departamento
""", engine)

dim_macrosector_sql = pd.read_sql("""
SELECT id_macrosector, macrosector
FROM dim_macrosector
""", engine)

In [54]:
fact = gold.copy()

Asociar IDS a las dimensiones 

In [55]:
fact = fact.merge(
    dim_empresa_sql,
    on="nit",
    how="left"
)

In [56]:
fact = fact.merge(
    dim_region_sql,
    left_on="región",
    right_on="region",
    how="left"
)

In [57]:
fact = fact.merge(
    dim_departamento_sql,
    left_on="departamento_domicilio",
    right_on="departamento",
    how="left"
)

In [58]:
fact = fact.merge(
    dim_macrosector_sql,
    on="macrosector",
    how="left"
)

In [59]:
print("Empresas sin ID:", fact["id_empresa"].isna().sum())
print("Regiones sin ID:", fact["id_region"].isna().sum())
print("Departamentos sin ID:", fact["id_departamento"].isna().sum())
print("Macrosectores sin ID:", fact["id_macrosector"].isna().sum())

Empresas sin ID: 0
Regiones sin ID: 0
Departamentos sin ID: 0
Macrosectores sin ID: 0


Construcción tabla de hechos

In [60]:
fact_final = fact[
    [
        "id_empresa",
        "id_region",
        "id_departamento",
        "id_macrosector",
        "ingresos_operacionales_2025",
        "ganancia_pérdida_2025",
        "total_activos_2025",
        "total_pasivos_2025",
        "total_patrimonio_2025",
        "crecimiento_ingresos_pct",
        "margen_utilidad_pct",
        "endeudamiento_pct",
        "roa_pct",
        "roe_pct"
    ]
].copy()

In [61]:
fact_final.rename(
    columns={
        "ingresos_operacionales_2025": "ingresos_2025",
        "ganancia_pérdida_2025": "utilidad_2025",
        "total_activos_2025": "activos_2025",
        "total_pasivos_2025": "pasivos_2025",
        "total_patrimonio_2025": "patrimonio_2025",
    },
    inplace=True
)

In [62]:
fact_final.info()

fact_final.head()

fact_final.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id_empresa                10000 non-null  int64  
 1   id_region                 10000 non-null  int64  
 2   id_departamento           10000 non-null  int64  
 3   id_macrosector            10000 non-null  int64  
 4   ingresos_2025             10000 non-null  int64  
 5   utilidad_2025             10000 non-null  int64  
 6   activos_2025              10000 non-null  int64  
 7   pasivos_2025              10000 non-null  int64  
 8   patrimonio_2025           10000 non-null  int64  
 9   crecimiento_ingresos_pct  10000 non-null  float64
 10  margen_utilidad_pct       10000 non-null  float64
 11  endeudamiento_pct         10000 non-null  float64
 12  roa_pct                   10000 non-null  float64
 13  roe_pct                   10000 non-null  float64
dtypes: float64(5), int

id_empresa                  0
id_region                   0
id_departamento             0
id_macrosector              0
ingresos_2025               0
utilidad_2025               0
activos_2025                0
pasivos_2025                0
patrimonio_2025             0
crecimiento_ingresos_pct    0
margen_utilidad_pct         0
endeudamiento_pct           0
roa_pct                     0
roe_pct                     0
dtype: int64

In [66]:
import pyodbc
import numpy as np

# Conexión a SQL Server
conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=LAPTOP-KJGERIIR;"
    r"DATABASE=EmpresasColombianas;"
    r"Trusted_Connection=yes;"
)

cursor = conn.cursor()

# Acelera muchísimo la inserción
cursor.fast_executemany = True

# Reemplazar NaN por None para SQL Server
fact_insert = fact_final.replace({np.nan: None})

# Convertir DataFrame en lista de tuplas
datos = list(fact_insert.itertuples(index=False, name=None))

# Consulta INSERT
sql = """
INSERT INTO dbo.fact_indicadores_financieros
(
    id_empresa,
    id_region,
    id_departamento,
    id_macrosector,
    ingresos_2025,
    utilidad_2025,
    activos_2025,
    pasivos_2025,
    patrimonio_2025,
    crecimiento_ingresos_pct,
    margen_utilidad_pct,
    endeudamiento_pct,
    roa_pct,
    roe_pct
)
VALUES
(
    ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?
)
"""

cursor.executemany(sql, datos)

conn.commit()

print(f"Se insertaron {len(datos)} registros correctamente.")

cursor.close()
conn.close()

Se insertaron 10000 registros correctamente.
